In [4]:
!pip install datasets transformers tokenizer accelerate

In [5]:
from datasets import load_dataset
import os

In [6]:
data = load_dataset('text', data_files='/content/general_knowledge_facts.txt')

Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
data

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 10425
    })
})

In [8]:
from transformers import (
    GPT2TokenizerFast,
    GPT2Config,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

In [9]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [10]:
tokenizer.pad_token=tokenizer.eos_token

In [11]:
def tokenize_function(examples):
  return tokenizer(examples["text"])

In [12]:
tokenized_data=data.map(tokenize_function, batched=True, num_proc=4, remove_columns=["text"])

Map (num_proc=4):   0%|          | 0/10425 [00:00<?, ? examples/s]

In [13]:
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 10425
    })
})

In [14]:
config=GPT2Config(
    vocab_size=tokenizer.vocab_size,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    n_layer=6,
    n_head=6,
    n_embd=384
)

new_model = GPT2LMHeadModel(config)

In [15]:
data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [20]:
training_args=TrainingArguments(
    output_dir="./gpt2-general-knowledge",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    eval_steps=50,
    save_total_limit=2
)

In [21]:
trainer=Trainer(
    model=new_model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_data["train"]
)

In [22]:
trainer.train()

Step,Training Loss
500,4.756564
1000,4.114390
1500,3.856197


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1630, training_loss=4.208028538241709, metrics={'train_runtime': 461.1701, 'train_samples_per_second': 113.028, 'train_steps_per_second': 3.534, 'total_flos': 424527800463360.0, 'train_loss': 4.208028538241709, 'epoch': 5.0})

In [35]:
new_model.save_pretrained("./gpt2-general-knowledge")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [36]:
tokenizer.save_pretrained("./gpt2-general-knowledge")

('./gpt2-general-knowledge/tokenizer_config.json',
 './gpt2-general-knowledge/tokenizer.json')

In [25]:
prompt="What is the largest species of shark?"

In [37]:
from transformers import pipeline

text_generator = pipeline("text-generation", model="./gpt2-general-knowledge", tokenizer="./gpt2-general-knowledge")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [38]:
output=text_generator(prompt, max_length=50, do_sample=True)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [39]:
print(output[0]['generated_text'])

What is the largest species of shark? in the potential in the world.\n" and the potential for its America.\n" in the potential?, or of the potential for the world?, and.\n", with the length from a chemical is therefore important to the country.\n\n" in the potential for the potential for the use of the potential.\n" in the rights of its surroundings, and to ensure that is important to the potential for the environment and the potential for their use of the potential for its surroundings, and or up to a to ensure that the world.\n\n" in the environment, and weighing an AI ethical dilemma solving bot, such as the British-.\n\n\n\n\n\n" in a molecule, the freedom.\n", and the potential for an AI ethical dilemma solving bot, such as the-being of the potential for its long, or to the potential for the role in the potential for its ethical dilemma solving bot, it is therefore important to the potential for the potential for the potential for its rights of the potential for its nervous, it is